# STAT 764 · Meeting 11 — Calibration

**Tuesday, October 13**

On Thursday a price decided the threshold. A \$150 home visit that prevents 5% of
readmissions is worth making when a patient's probability of coming back beats
**p\* = \$150 ÷ (0.05 × \$17,700) = 0.169** — two prices and a division, no search.

That rule takes the model at its word. It treats "0.20" as "20 of every 100 patients like
this one come back." If the true number were 12 of 100, every visit in that range would lose
money. And nothing we have measured so far would have warned us: AUC, the number we have
leaned on since Meeting 9, only checks whether the patients are in the right **order**.

**The question for today.** The model says 0.20. Do 20% of those patients come back — and
how would you check?

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit, logit
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The readmission model from Meetings 9 and 10, rebuilt so this notebook stands on its own.
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)

NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})

# Split by PATIENT, so nobody is on both sides -- the same split as Meetings 9 and 10.
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))


def readmission_model(**options):
    """The Meeting 9 pipeline. `options` go to LogisticRegression (e.g. C=0.001)."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000, **options)),
    ])


model = readmission_model().fit(X_all.iloc[train], y_all.iloc[train])
p = model.predict_proba(X_all.iloc[test])[:, 1]      # one probability per discharge
y = y_all.iloc[test].to_numpy()                       # 1 = back within 30 days

print(f"  {len(y):,} discharges the model never saw · readmitted within 30 days: {y.mean():.1%}")
print(f"  AUC {roc_auc_score(y, p):.3f} · probabilities from {p.min():.3f} to {p.max():.3f}")

Three helpers for today, each a few lines you have seen before. `net_per_1000` is Thursday's
money: dollars saved minus dollars spent per 1,000 discharges. `by_decile` is the check this
meeting is about. `reliability_plot` draws that check.

In [ ]:
READMISSION = 17_700           # dollars -- AHRQ HCUP Statistical Brief #307 (2024), 2020 data
P_STAR = 150 / (0.05 * READMISSION)    # Thursday's break-even for a $150 visit: 0.169


def net_per_1000(probs, outcome, threshold=P_STAR, cost=150, prevents=0.05):
    """Thursday's money: visit everyone at or above `threshold`. Saved minus spent, per 1,000."""
    visit = probs >= threshold
    saved = (visit & (outcome == 1)).sum() * prevents * READMISSION
    spent = visit.sum() * cost
    return (saved - spent) / len(outcome) * 1000


def by_decile(probs, outcome):
    """Ten equal groups by probability: what the model SAID against what HAPPENED in each."""
    group = pd.qcut(probs, 10, labels=False, duplicates="drop")
    return pd.DataFrame({"said": [probs[group == k].mean() for k in range(10)],
                         "came back": [outcome[group == k].mean() for k in range(10)]},
                        index=pd.RangeIndex(1, 11, name="tenth")).round(3)


def reliability_plot(versions, outcome, ax=None, title=""):
    """One point per tenth of patients: x = what the model said, y = what happened.
    A calibrated model's points sit on the dashed diagonal."""
    ax = ax or plt.subplots(figsize=(4.6, 4.6))[1]
    top = 0.0
    for name, probs in versions.items():
        table = by_decile(probs, outcome)
        ax.plot(table["said"], table["came back"], marker="o", label=name)
        top = max(top, table.to_numpy().max())
    top = min(1.0, top * 1.1)
    ax.plot([0, top], [0, top], ls="--", color="gray", label="calibrated")
    ax.set(xlim=(0, top), ylim=(0, top), xlabel="the model said (average in the tenth)",
           ylabel="actually came back (share)", title=title)
    ax.legend(loc="upper left", fontsize=8)
    return ax

## 1. AUC cannot see the numbers

AUC asks one question of every pair of patients — one who came back, one who did not: *is the
one who came back ranked higher?* Change the probabilities in any way that keeps their order
and every one of those answers stays the same, so AUC stays the same.

Squaring keeps the order: 0.3 is above 0.2, and 0.09 is above 0.04. So does a square root.

**Predict before you run it, and write it down:** square every probability. What happens to
AUC? What happens to Thursday's \$150 decision — visit everyone above 0.169?

In [ ]:
# Three versions of the same model. Each keeps the ORDER of the patients exactly,
# so each must have the same AUC. Only the numbers change.
versions = {"the model's p": p, "p squared": p ** 2, "square root of p": np.sqrt(p)}

print(f"  {'':20}{'AUC':>7}{'average p':>11}{'visits':>9}{'net per 1,000':>16}")
for name, probs in versions.items():
    print(f"  {name:20}{roc_auc_score(y, probs):>7.3f}{probs.mean():>11.3f}"
          f"{np.mean(probs >= P_STAR):>9.1%}{net_per_1000(probs, y):>16,.0f}")
print(f"  {'what happened':20}{'':>7}{y.mean():>11.3f}")

**What this shows.** One AUC, 0.655, three times — and three different decisions. The square
root says the average patient has a 33% chance when 11% come back: the \$150 rule visits almost
everyone and loses about \$52,000 per 1,000 discharges. Squaring says 2%: the rule visits almost
no one and leaves most of the money on the table.

AUC checks the **order**. Whether the **numbers** mean what they say is a separate property
with its own name — **calibration** — and it needs its own check.

## 2. The check: does 0.2 mean 20%?

Sort the patients into ten equal groups by probability. In each group, compare the average
probability the model gave (**said**) with the share who actually came back (**came back**). If
the model is **calibrated**, the two columns match, and on a plot of one against the other the
points sit on the diagonal. The plot is called a **reliability diagram** (or calibration plot).

It is the decile table from Thursday's notebook, Section 3 — here for the model and for the
square-root version side by side.

In [ ]:
# The check, as a table: each row is a tenth of the 24,706 test discharges, lowest
# probabilities first. Left: the model as fitted. Right: the square-root version.
print(pd.concat({"the model's p": by_decile(p, y),
                 "square root of p": by_decile(np.sqrt(p), y)}, axis=1).to_string())

# The same check, as a picture: on the diagonal = the numbers mean what they say.
reliability_plot({"the model's p": p, "square root of p": np.sqrt(p)}, y,
                 title="Same AUC, 0.655")
plt.tight_layout()
plt.show()

# The quickest single check: the average probability against the share who came back.
print(f"  on average the model said {p.mean():.3f}; {y.mean():.3f} came back")

**What this shows.** The model's own probabilities sit close to the diagonal: within about a
point in every tenth except the top, where it says 0.273 and 0.248 come back. So when this
model says 0.2, about 20% come back, and Thursday's \$150 rule was entitled to use its numbers.
The square-root version has exactly the same *came back* column — same patients, same order —
and a *said* column that is too high everywhere.

The last line is the first check anyone should run: **average said against average happened**,
0.115 against 0.111. Van Calster et al. (2019), today's reading, call it
**calibration-in-the-large**.

## 3. Three ways the numbers go wrong

Miscalibration comes in three shapes. Each panel below is made from our model by changing its
numbers but never its order, so **all three still have AUC 0.655**.

| shape | on the reliability diagram | what usually causes it |
|---|---|---|
| **too high (or too low) everywhere** | the points sit *below* the diagonal when the numbers are too high (the model says more than happens), above it when too low | the rate of the outcome changed since the model was fitted — or the classes were reweighted (Thursday's notebook, Section 3) |
| **too extreme** — too sure of itself | flatter than the diagonal: too low at the bottom, too high at the top | **overfitting**: too many parameters for too little data |
| **too timid** | steeper than the diagonal: too high at the bottom, too low at the top | **too much shrinkage**: a penalty so strong it pulls every patient toward the average |

**One number for the spread: the calibration slope.** Take patients the model did not see, and
fit a logistic regression of what happened on the model's own log-odds. If the spread of the
numbers is right, the slope is 1. Below 1, the model is too extreme — its log-odds should be
shrunk toward the middle. Above 1, it is too timid. Van Calster et al. (2019), today's reading,
report it for every model, beside the averages. Each panel below shows its slope, and its average
said against the share that came back. Our own model comes first, as the yardstick.

In [ ]:
def calibration_slope(probs, outcome):
    """Refit what happened on the model's own log-odds, in patients it did not see.
    The slope: 1 = the spread is right; below 1 = too extreme; above 1 = too timid."""
    model_log_odds = logit(np.clip(probs, 1e-6, 1 - 1e-6)).reshape(-1, 1)
    refit = LogisticRegression(C=1e6, max_iter=1000).fit(model_log_odds, outcome)  # ~no penalty
    return refit.coef_[0, 0]


# Work on the log-odds scale, where "shift" and "stretch" are clean.
log_odds = logit(p)
middle = log_odds.mean()

shapes = {
    "too high everywhere": expit(log_odds + 0.6),                  # shift every patient up
    "too extreme": expit(middle + 2.0 * (log_odds - middle)),      # stretch away from the middle
    "too timid": expit(middle + 0.5 * (log_odds - middle)),        # squeeze toward the middle
}

# Our model first, as the yardstick; then the three shapes. Each panel: the reliability check,
# with its AUC and slope on top and its averages in the corner.
panels = {"our model": p, **shapes}
fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
for ax, (name, probs) in zip(axes, panels.items()):
    title = f"{name}\nAUC {roc_auc_score(y, probs):.3f} · slope {calibration_slope(probs, y):.2f}"
    reliability_plot({name: probs}, y, ax=ax, title=title)
    ax.text(0.97, 0.04, f"average said {probs.mean():.3f}\ncame back {y.mean():.3f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5)
plt.tight_layout()
plt.show()

**What this shows.** Three different ways to be wrong about every patient, and AUC — 0.655 on all
three — cannot tell them apart from our model, in the first panel. The slope finds two of them:
*too extreme* is 0.47 and *too timid* 1.88, against 0.94 for our model. But *too high everywhere* has our model's
slope, 0.94: shifting every patient by the same amount leaves the spread alone. Its corner says
what is wrong — 0.186 said, 0.111 came back. **The averages check the level; the slope checks the
spread.** You need both.

**Reading the slope in log-odds.** Logistic regression works in log-odds, log(p / (1 − p)), and a
difference in log-odds between two patients is the log of their **odds ratio**. So the slope
says how much of the model's difference is real: when the model says one patient's odds are
**twice** another's, the true ratio is about **2 to the power of the slope**. Our model:
2^0.94 ≈ 1.9, close to the 2 it claims. *Too extreme*, made by doubling our model's log-odds
differences — so its slope is half of ours: 2^0.47 ≈ 1.4; it exaggerates every difference.
*Too timid*: 2^1.88 ≈ 3.7; it understates them.

The studio puts real causes behind two of the shapes: a model fitted on last year's patients,
and models that are too sure or too timid.

### From the lab — scores are not probabilities

The breast ultrasound model from Meeting 9 reads about 12,400 exams from Mayo Clinic, 8% of them
cancers, with AUC 0.93. To catch 99%, 98%, 95%, 90%, 80% or 70% of the cancers, its cut-offs
are **0.02, 0.04, 0.07, 0.13, 0.52 and 0.86**.

Those are **scores**, not probabilities. A cut-off of 0.13 does not mean "a 13% chance of
cancer" — it is simply the score that happens to catch 90% of the cancers. For choosing a
cut-off that is fine: only the order matters, exactly as for AUC. But if the system printed
"13%" next to an image for a radiologist to read, that number would have to be *calibrated*,
checked the way Section 2 checks ours, because a person would act on it as a probability.

## Studio

Two questions to finish, the third if you have time. Every question uses the same **two
checks** from today, then Thursday's money:

- **the order:** `roc_auc_score(outcome, probs)`
- **the numbers:** `by_decile(probs, outcome)` — the table — and, if you like, the picture:
  `reliability_plot({"a name": probs}, outcome)`
- **the \$150 decision:** the share of patients visited, `np.mean(probs >= P_STAR)`, and
  `net_per_1000(probs, outcome)`

`outcome` is always the 0/1 outcome **for the same patients** as `probs` — for the test patients
that is `y`, as in Sections 1–3.

**Before each question, write your guess** as a comment in its cell, in Section 3's words: will
the numbers be *too high everywhere*, *too low everywhere*, *too extreme*, *too timid*, or *fine*?
Keep the guess when the answer disagrees — the Compare on Thursday uses it.

The notebook is due 24 hours after class. Done early? Finish Lab 5 (due tonight), then
Mini-project 1.

### 1. A risk percentile

Some vendors report risk as a percentile: 0.83 means "riskier than 83% of patients." Make one
from our model's `p`:

```python
from scipy.stats import rankdata
pct = rankdata(p) / len(p)        # ranks 1 to 24,706, divided so they run from 0 to 1
```

- **(a)** Give `p` and `pct` both checks, side by side.
- **(b)** A nurse manager treats `pct` as a probability and applies Thursday's \$150 rule: visit
  everyone at or above 0.169. What share of patients get a visit, and what is the net per 1,000
  discharges? Compare with the same rule on `p`.
- **(c)** Thursday's other decision: a nurse who can review only the **top 5%** of discharges.
  Take the top 5% by each score — `pct >= np.quantile(pct, 0.95)`, and the same for `p`. Are they
  the same patients? How many readmissions does each queue reach?

**Answer in a sentence:** why does the percentile ruin the \$150 rule but not the queue?

### 2. Last year's model, this year's patients

The cell just below these questions picks two groups of patients by **when they were first
seen** (`encounter_id` rises with time), with a gap between them, as there would be between
building a model and using it: `earlier` (fit on these) and `later` (score these). It does not
fit anything — that is your job.

- **(a)** Fit `readmission_model()` on `X_all.iloc[earlier]` and `y_all.iloc[earlier]`. Score the
  later patients: `p_late` from `X_all.iloc[later]`, with outcome
  `y_late = y_all.iloc[later].to_numpy()`.
- **(b)** Give it both checks — with `y_late`, not `y`. Are its numbers right for the later
  patients?
- **(c)** Find the patients it would visit under the \$150 rule who sit **just above the line**:
  `near = (p_late >= P_STAR) & (p_late < 0.25)`. How many are there? What did the model say they
  would do (`p_late[near].mean()`), and what share actually came back (`y_late[near].mean()`)?

**Answer in a sentence:** is a \$150 visit worth it for these patients? A visit saves
0.05 × \$17,700 = **\$885** for each patient who would have come back, so it is worth \$150 only if
more than 0.169 of them come back.

### 3. Too sure, or too timid? *(if you have time)*

In scikit-learn, `C` is the **inverse** of the penalty: a small `C` is a strong penalty, a large
`C` almost none. Fit the pipeline twice, and score the usual test patients (`X_all.iloc[test]`,
outcome `y`):

- `readmission_model(C=0.0001)` on all of `train` — a very strong penalty;
- `readmission_model(C=10_000)` on only 2,000 training discharges — almost no penalty, and few
  patients: `few = np.random.default_rng(0).choice(train, 2000, replace=False)`, then
  `X_all.iloc[few]` and `y_all.iloc[few]`.

Give each both checks and the \$150 decision.

**Answer in a sentence:** which one is too extreme and which too timid? (Compare how widely the
*said* column spreads with how widely *came back* spreads.) Which would you rather hand the nurse?

In [ ]:
# Question 2's data -- run as is. Patients are placed by the date of their FIRST stay
# (encounter_id rises with time), so each patient is on one side only.
first_seen = enc.groupby("patient_nbr")["encounter_id"].transform("min")
early_cut, late_cut = np.quantile(enc["encounter_id"], [0.60, 0.75])
earlier = np.where(first_seen <= early_cut)[0]       # fit on these
later = np.where(first_seen > late_cut)[0]           # score these
print(f"  fit on {len(earlier):,} stays (readmitted {y_all.iloc[earlier].mean():.1%}) · "
      f"score {len(later):,} later stays (readmitted {y_all.iloc[later].mean():.1%})")

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

## Compare — at the start of Thursday's class

We compare answers at the start of Thursday's class, built from the notebooks you upload. So
leave these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 1: what the \$150 rule does with the percentile — and why the review queue does not
   care.
3. Question 2: what the model said and what happened for the patients just above the line, and
   whether a visit is worth it for them.

## Exit ticket

> A colleague says: "Our model has AUC 0.80, so its probabilities are trustworthy." In two or
> three sentences, what would you tell them — and what one check would you run?

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio
notebook — Day 11". Your two studio points need this answered: a blank exit ticket earns
zero. It is not graded for being right — it is how I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Van Calster et al. (2019), "Calibration: the Achilles heel of predictive analytics,"**
*BMC Medicine* 17:230 — today's reading. Its levels of calibration (mean, weak, moderate,
strong) put names to what Sections 2 and 3 checked.

📗 **The Brier score.** Lab 5 introduced it: the average squared distance between each
probability and what happened. It sees the order *and* the numbers at once. Thursday takes it
apart.

📗 **Recalibration** — Platt scaling, isotonic regression. If the order is good and the numbers
are off, you can fix the numbers without touching the order. Thursday, with the one rule that
matters: the fix must be learned on patients the model was not fitted on.